In [1]:
import numpy as np
import pandas as pd
import subprocess

In [2]:
# presets, functions

# preset values for the analysis:
#____________________________________________________________________________________________________________________
# physical constants
Z = 6
A = 12
mass_nucleon = 0.938273
mass_nucleus = A * 0.931494
alpha_fine = 1 / 137
Ex_cut = 0.03
Ex_cut_lowq = 0.1

# three-momentum bin centers
qvcenters = [0.100, 0.148, 0.167, 0.205, 0.240, 0.300, 0.380, 0.475, 0.570, 0.649, 0.756, 0.991, 1.619, 1.921, 2.213, 2.500, 2.783, 3.500]
# three-momentum edges
qvbins = [0.063, 0.124, 0.158, 0.186, 0.223, 0.270, 0.340, 0.428, 0.523, 0.609, 0.702, 0.878, 1.302, 1.770, 2.067, 2.357, 2.642, 2.923, 4.500]
# four-momentum squared bin names, in string format
qvbin_names = ['[0.063,0.124]', '[0.124,0.158]', '[0.158,0.186]', '[0.186,0.223]', '[0.223,0.270]', '[0.270,0.340]', '[0.340,0.428]', '[0.428,0.523]', '[0.523,0.609]',
                '[0.609,0.702]', '[0.702,0.878]', '[0.878,1.302]', '[1.302,1.770]', '[1.770,2.067]', '[2.067,2.357]', '[2.357,2.642]', '[2.642,2.923]', '[2.923,4.500]']

# four-momentum squared bin centers
Q2centers = [0.010, 0.020, 0.026, 0.040, 0.056, 0.093, 0.120, 0.160, 0.265, 0.380, 0.500, 0.800, 1.250, 1.750, 2.250, 2.750, 3.250, 3.750]
# four-momentum squared bin edges
Q2bins = [0.004, 0.015, 0.025, 0.035, 0.045, 0.070, 0.100, 0.145, 0.206, 0.322, 0.438, 0.650, 1.050, 1.500, 2.000, 2.500, 3.000, 3.500, 4.000]
# four-momentum squared bin names, in string format
Q2bin_names = ['[0.004,0.015]', '[0.015,0.025]', '[0.025,0.035]', '[0.035,0.045]', '[0.045,0.070]', '[0.070,0.100]', '[0.100,0.145]', '[0.145,0.206]', '[0.206,0.322]',
                '[0.322,0.438]', '[0.438,0.650]', '[0.650,1.050]', '[1.050,1.500]', '[1.500,2.000]', '[2.000,2.500]', '[2.500,3.000]', '[3.000,3.500]', '[3.500,4.000]']

dataSet_to_normalization = {1: 0.95971, 2: 0.96416, 3: 1.0744, 4: 0.99482, 5: 0.93381, 6: 1.0126, 
                            7: 0.96716, 8: 1.0238, 9: 0.97904, 10: 0.99064, 11: 0.98384, 12: 1.0000, 
                            13: 1.0163, 14: 1.0300, 15: 1.0190, 16: 0.95853, 17: 1.0174, 18: 1.0168, 
                            19: 1.0794, 20: 1.0000, 21: 0.9500, 22: 1.1095, 23: 0.9310, 24: 1.0019, 
                            25: 0.8500, 26: 1.0000, 33: 0.9980, 34: 0.9677, 35: 0.9561}

dataSet_to_normError = {1: 0.62926E-02, 2: 0.12908E-01, 3: 0.80983E-02, 4: 0.69809E-02, 5: 0.16758E-01, 6: 0.92261E-02, 
                        7: 0.15546E-01, 8: 0.65203E-02, 9: 0.55606E-02, 10: 0.75245E-02, 11: 0.25318E-01, 12: 0.0, 
                        13: 0.17632E-02, 14: 0.91993E-02, 15: 0.63181E-02, 16: 0.25582E-01, 17: 0.42184E-01, 18: 0.68067E-01, 
                        19: 0.35847E-01, 20: 0.0, 21: 0.25, 22: 0.1, 23: 0.1, 24: 0.184E-01, 
                        25: 0.02, 26: 0.02, 33: 0.415E-01, 34: 0.173E-01, 35: 0.231E-01}

In [3]:
# read dataframe from csv file
def prepare_df(df):

    # calculate normalized cross section:
    if 'error' not in df.columns:
        df['error'] = df['cross'] * 0.02
    if 'dataSet' not in df.columns:
        df['dataSet'] = -1
        df['normalization'] = 1.0
        df['normError'] = 0.0
    else:
        df["normalization"] = df["dataSet"].map(dataSet_to_normalization)
        df["normError"] = df["dataSet"].map(dataSet_to_normError)
    df['system_err'] = 0.0
    df['normCross'] = df['cross'] * df['normalization']
    df['error'] = np.sqrt(df['error']**2 + ((df['system_err'] * df['cross'])**2))
    df['normCrossError'] = df['normCross'] * np.sqrt((df['error'] / df['cross'])**2 + (df['normError'] / df['normalization'])**2)
    print(df.loc[df['normalization'] == 1, 'dataSet'].unique())
    
    # calculate the kinematic variables:
    df["Veff"] = 0.0031
    df["ThetaRad"] = df["ThetaDeg"] * np.pi / 180
    df["sin2(T/2)"] = (np.sin(df["ThetaRad"] / 2))**2
    df["cos2(T/2)"] = (np.cos(df["ThetaRad"] / 2))**2
    df["tan2(T/2)"] = (np.tan(df["ThetaRad"] / 2))**2
    df["Ex"] = df["nu"] - (df["E0"] - df["E0"] / (1 + 2 * df["E0"] * df["sin2(T/2)"] / mass_nucleus))
    df["W2original"] = mass_nucleon**2 + 2 * mass_nucleon * df["nu"] - (4 * df["E0"] * (df["E0"] + df["Veff"]) * df["sin2(T/2)"])
    df["Ffoc2"] = ((df["E0"] + df["Veff"]) / df["E0"])**2
    df["Ep"] = df["E0"] - df["nu"]
    # ______________________starting here: effective values only______________________
    df["E0original"] = df["E0"]
    df["E0"] = df["E0"] + df["Veff"]
    df["Ep"] = df["Ep"] + df["Veff"]
    df["R"] = 1.1 * (df["A"])**(1/3) + 0.86 / ((df["A"])**(1/3))
    df["Q2"] = 4 * df["E0"] * (df["Ep"]) * df["sin2(T/2)"]
    df["qv2"] = df["nu"]**2 + df["Q2"]
    df["qv"] = np.sqrt(df["qv2"])
    df["W2"] = mass_nucleon**2 + 2 * mass_nucleon * df["nu"] - df["Q2"]
    df["epsilon"] = 1 / (1 + 2 * (1 + (df["nu"]**2) / df["Q2"]) * df["tan2(T/2)"])
    df["gamma"] = alpha_fine * df["Ep"] * (df["W2"] - mass_nucleon**2) / (( 4 * ((np.pi)**2) * df["Q2"] * mass_nucleon * df["E0"]) * (1 - df["epsilon"]))
    df["Sig_R"] = df["normCross"] / df["gamma"]
    df["D_sig_R"] = df["error"] / df["gamma"]
    df["Sig_mott"] = df["Ffoc2"] * alpha_fine**2 * df["cos2(T/2)"] * (2 * df["E0"] * df["sin2(T/2)"])**-2

    # calculate the Rosenbluth quantity:
    df["Hcc"] = ((df["qv"]**4) / (4 * (alpha_fine**2) * (df["Ep"]**2) * (df["cos2(T/2)"] + 2 * (df["qv2"] / df["Q2"]) * df["sin2(T/2)"]))) / df["Ffoc2"]
    df["Hcc_Sig(nb)"] = df["Hcc"] * df["normCross"]
    df["Hcc_error(nb)"] = df["Hcc"] * df["normCrossError"]
    df["Hcc_Sig(GeV)"] = df["Hcc_Sig(nb)"] / ((0.1973269**2) * 10000000)
    df["Hcc_error(GeV)"] = df["Hcc_error(nb)"] / ((0.1973269**2) * 10000000)

    # subdivide the data into bins
    df['qvbin'] = 0
    df['qvcenter'] = 0
    df["qvbin"] = pd.cut(x=df["qv"], bins = qvbins, labels = qvbin_names, right=True)
    df["qvcenter"] = pd.cut(x=df["qv"], bins = qvbins, labels = qvcenters, right=True)
    df['qvcenter'] = pd.to_numeric(df['qvcenter'])
    df['Q2bin'] = 0
    df['Q2center'] = 0
    df["Q2bin"] = pd.cut(x = df["Q2"], bins = Q2bins, labels = Q2bin_names, right = True)
    df["Q2center"] = pd.cut(x = df["Q2"], bins = Q2bins, labels = Q2centers, right = True)
    df['Q2center'] = pd.to_numeric(df['Q2center'])
    df = df.dropna()

    # bin-centering related:
    df['Exbin_qv'] = 0.0
    df['Excenter_qv'] = 0.0
    df['nucenter_ex_qv'] = 0.0
    df['epcenter_ex_qv'] = 0.0

    df['W2bin_qv'] = 0.0
    df['W2center_qv'] = 0.0
    df['nucenter_w2_qv'] = 0.0
    df['epcenter_w2_qv'] = 0.0

    df['Exbin_q2'] = 0.0
    df['Excenter_q2'] = 0.0
    df['nucenter_ex_q2'] = 0.0
    df['epcenter_ex_q2'] = 0.0

    df['W2bin_q2'] = 0.0
    df['W2center_q2'] = 0.0
    df['nucenter_w2_q2'] = 0.0
    df['epcenter_w2_q2'] = 0.0

    def Exedges_epsilon_range(df = None, edges = None, min_range = 0.25):
        i = 0
        while i < len(edges) - 2:
            lo, hi = edges[i], edges[i + 2]
            # Calculate y-range in the merged bin [lo, hi)
            sub = df[(df['Ex'] >= lo) & (df['Ex'] < hi)]
            if not sub.empty and (sub['epsilon'].max() - sub['epsilon'].min()) < min_range:
                # Merge by removing mid-edge
                edges = np.delete(edges, i + 1)
                # Step back to re-check previous merge
                if i > 0: i -= 1
            else:
                i += 1
        return edges

    def W2edges_epsilon_range(df = None, edges = None, min_range = 0.25):
        i = 0
        while i < len(edges) - 2:
            lo, hi = edges[i], edges[i + 2]
            # Calculate y-range in the merged bin [lo, hi)
            sub = df[(df['W2'] >= lo) & (df['W2'] < hi)]
            if not sub.empty and (sub['epsilon'].max() - sub['epsilon'].min()) < min_range:
                # Merge by removing mid-edge
                edges = np.delete(edges, i + 1)
                # Step back to re-check previous merge
                if i > 0: i -= 1
            else:
                i += 1
        return edges
    
    W2ns = np.array([30,19,16,14,20,
                22,22,21,21,18,
                20,21,25,20,18,
                18,20,28])

    for i in range(len(qvcenters)):
        qvcenter = qvcenters[i]

        # Ex < 30MeV:
        mask = (df['qvcenter'] == qvcenter) & (df['Ex'] < Ex_cut)
        if qvcenter == 0.1:
            mask = (df['qvcenter'] == qvcenter) & (df['Ex'] < Ex_cut_lowq)
        if len(df.loc[mask, 'Ex']) > 0:
            n_bins = max(1, len(df.loc[mask, 'Ex']) // 5)
            Exedges = np.quantile(df.loc[mask, 'Ex'], np.linspace(0, 1, n_bins + 1))
            Exedges = Exedges_epsilon_range(df = df.loc[mask], edges = Exedges)
            Excenters = (Exedges[:-1] + Exedges[1:]) / 2
            df.loc[mask, 'Exbin_qv'] = pd.cut(df.loc[mask, 'Ex'], bins = Exedges, labels = False, include_lowest = True, duplicates = 'drop')
            df.loc[mask, 'Excenter_qv'] = df.loc[mask, 'Exbin_qv'].map(lambda i: Excenters[int(i)] if pd.notnull(i) else np.nan)
            df.loc[mask, 'nucenter_ex_qv'] = np.sqrt(mass_nucleus**2 + qvcenter**2 + 2 * mass_nucleus * df.loc[mask, 'Excenter_qv']) - mass_nucleus
            df.loc[mask, 'epcenter_ex_qv'] = 1 / (1 + 2 * (1 + (df.loc[mask, 'nucenter_ex_qv']**2) / (qvcenter**2 - df.loc[mask, 'nucenter_ex_qv']**2)) * df.loc[mask, 'tan2(T/2)'])

        # Ex >= 30MeV:
        mask = (df['qvcenter'] == qvcenter) & (df['Ex'] >= Ex_cut)
        if qvcenter == 0.1:
            mask = (df['qvcenter'] == qvcenter) & (df['Ex'] >= Ex_cut_lowq)
        if len(df.loc[mask, 'W2']) > 0:
        # if qvcenter > 0.1:
            n_bins = max(1,len(df.loc[mask, 'W2']) // W2ns[i])
            W2edges = np.quantile(df.loc[mask, 'W2'], np.linspace(0, 1, n_bins + 1))
            W2edges = W2edges_epsilon_range(df = df.loc[mask], edges = W2edges)
            if qvcenter == 0.38: # good
                W2edges = np.delete(W2edges, [-2, -3, -4])
            if qvcenter == 0.475: # good
                W2edges = np.delete(W2edges, [-2, -3, -4, -5])
            if qvcenter == 0.57: # good
                W2edges = np.delete(W2edges, [-2, -3, -4, -5, -6, -7, -8])
            if qvcenter == 0.649: # good
                W2edges = np.delete(W2edges, [-2, -3, -7])
            if qvcenter == 0.756:
                W2edges = np.delete(W2edges, [-9])
            W2centers = (W2edges[:-1] + W2edges[1:]) / 2
            df.loc[mask, 'W2bin_qv'] = pd.cut(df.loc[mask, 'W2'], bins = W2edges, labels = False, include_lowest = True)
            df.loc[mask, 'W2center_qv'] = df.loc[mask, 'W2bin_qv'].map(lambda i: W2centers[int(i)] if pd.notnull(i) else np.nan)
            df.loc[mask, 'nucenter_w2_qv'] = np.sqrt(qvcenter**2 + df.loc[mask, 'W2center_qv']) - mass_nucleon
            df.loc[mask, 'epcenter_w2_qv'] = 1 / (1 + 2 * (1 + (df.loc[mask, 'nucenter_w2_qv']**2) / (qvcenter**2 - df.loc[mask, 'nucenter_w2_qv']**2)) * df.loc[mask, 'tan2(T/2)']) 

    W2ns = np.array([20,23,19,26,26,
                30,22,23,25,24,
                20,22,20,22,20,
                19,17,18])

    for i in range(len(Q2centers)):
        Q2center = Q2centers[i]

        # Ex < 30MeV:
        mask = (df['Q2center'] == Q2center) & (df['Ex'] < Ex_cut)
        if Q2center == 0.01:
            mask = (df['Q2center'] == Q2center) & (df['Ex'] < Ex_cut_lowq)
        if len(df.loc[mask, 'Ex']) > 0:
            n_bins = max(1,len(df.loc[mask, 'Ex']) // 5)
            Exedges = np.quantile(df.loc[mask, 'Ex'], np.linspace(0, 1, n_bins + 1))
            Exedges = Exedges_epsilon_range(df = df.loc[mask], edges = Exedges)
            Excenters = (Exedges[:-1] + Exedges[1:]) / 2
            df.loc[mask, 'Exbin_q2'] = pd.cut(df.loc[mask, 'Ex'], bins=Exedges, labels=False, include_lowest=True,duplicates='drop')
            df.loc[mask, 'Excenter_q2'] = df.loc[mask, 'Exbin_q2'].map(lambda i: Excenters[int(i)] if pd.notnull(i) else np.nan)
            df.loc[mask, 'nucenter_ex_q2'] = df.loc[mask, 'Excenter_q2'] + Q2center / (2 * mass_nucleus)
            df.loc[mask, 'epcenter_ex_q2'] = 1 / (1 + 2 * (1 + (df.loc[mask, 'nucenter_ex_q2']**2) / Q2center) * df.loc[mask, 'tan2(T/2)']) 

        # Ex >= 30MeV:
        mask = (df['Q2center'] == Q2center) & (df['Ex'] >= Ex_cut)
        if Q2center == 0.01:
            mask = (df['Q2center'] == Q2center) & (df['Ex'] > Ex_cut_lowq)
        if len(df.loc[mask, 'W2']) > 0:
            n_bins = max(1, len(df.loc[mask, 'W2']) // W2ns[i])
            W2edges = np.quantile(df.loc[mask, 'W2'], np.linspace(0, 1, n_bins + 1))
            W2edges = W2edges_epsilon_range(df = df.loc[mask], edges = W2edges)
            if Q2center == 0.02: # good
                W2edges = np.delete(W2edges, [-3])
            if Q2center == 0.026: # good
                W2edges = np.delete(W2edges, [-2,-3])
            if Q2center == 0.04: # good
                W2edges = np.delete(W2edges, [-2])
            if Q2center == 0.056: # good
                W2edges = np.delete(W2edges, [1,3,5,-2,-3,-4,-5])
            if Q2center == 0.093: # good
                W2edges = np.delete(W2edges, [2,4,9,-2,-3,-4])
            if Q2center == 0.12: # good
                W2edges = np.delete(W2edges, [-2,-3,-5,-6,-7])
            if Q2center == 0.16: # good
                W2edges = np.delete(W2edges, [-3])
            if Q2center == 0.265: # good
                W2edges = np.delete(W2edges, [-2])
            if Q2center == 0.5: # good
                W2edges = np.delete(W2edges, [-7])
            W2centers = (W2edges[:-1] + W2edges[1:]) / 2
            df.loc[mask, 'W2bin_q2'] = pd.cut(df.loc[mask, 'W2'], bins = W2edges, labels = False, include_lowest = True, duplicates = 'drop')
            df.loc[mask, 'W2center_q2'] = df.loc[mask, 'W2bin_q2'].map(lambda i: W2centers[int(i)] if pd.notnull(i) else np.nan)
            df.loc[mask, 'nucenter_w2_q2'] = (df.loc[mask, 'W2center_q2'] - mass_nucleon**2 + Q2center) / (2 * mass_nucleon)
            df.loc[mask, 'epcenter_w2_q2'] = 1 / (1 + 2 * (1 + (df.loc[mask, 'nucenter_w2_q2']**2) / Q2center) * df.loc[mask, 'tan2(T/2)']) 

    df = df.dropna()
    return df

In [4]:
def calculate_bc(df):
    response_columns = ['i','RTTOT','RLTOT','RTnoNS','RLnoNS']

    # calculate bc_qv_ex
    df[['qvcenter','Excenter_qv']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_qv_ex.exe', 'input.txt'], stdout=output_file) 
    subprocess.run(['sleep', '0.5'])
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_qvc_ex'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_qvc_ex'] = df.index.map(df_response.set_index('i')['RTTOT'])

    df[['qv','Ex']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_qv_ex.exe', 'input.txt'], stdout=output_file) 
    subprocess.run(['sleep', '0.5'])
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_qvd_ex'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_qvd_ex'] = df.index.map(df_response.set_index('i')['RTTOT'])

    df['bc_qv_ex'] = 1.0
    for qvcenter in qvcenters:
        # Ex < 30MeV:
        mask = (df['qvcenter'] == qvcenter) & (df['Ex'] < Ex_cut)
        if qvcenter == 0.1:
            mask = (df['qvcenter'] == qvcenter) & (df['Ex'] < Ex_cut_lowq)
        df.loc[mask, 'bc_qv_ex'] = (df.loc[mask, 'epcenter_ex_qv'] * df.loc[mask, 'RL_qvc_ex'] + 0.5 * ((qvcenter**2) / (qvcenter**2 - df.loc[mask,'nucenter_ex_qv']**2)) * df.loc[mask, 'RT_qvc_ex']) / (df.loc[mask, 'epsilon'] * df.loc[mask, 'RL_qvd_ex'] + 0.5 * (df.loc[mask, 'qv2'] / df.loc[mask, 'Q2']) * df.loc[mask, 'RT_qvd_ex'])
    print('RL RT bc_q2_ex done.')

    # calculate bc_qv_w2
    df[['qvcenter','W2center_qv']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_qv_w2.exe', 'input.txt'], stdout=output_file) 
    subprocess.run(['sleep', '0.5'])
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_qvc_w2'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_qvc_w2'] = df.index.map(df_response.set_index('i')['RTTOT'])

    df[['qv','W2']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_qv_w2.exe', 'input.txt'], stdout=output_file) 
    subprocess.run(['sleep', '0.5'])
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_qvd_w2'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_qvd_w2'] = df.index.map(df_response.set_index('i')['RTTOT'])

    df['bc_qv_w2'] = 1.0
    for qvcenter in qvcenters:
        # Ex >= 30MeV:
        mask = (df['qvcenter'] == qvcenter) & (df['Ex'] >= Ex_cut)
        df.loc[mask, 'bc_qv_w2'] = (df.loc[mask, 'epcenter_w2_qv'] * df.loc[mask, 'RL_qvc_w2'] + 0.5 * ((qvcenter**2) / (qvcenter**2 - df.loc[mask,'nucenter_w2_qv']**2)) * df.loc[mask, 'RT_qvc_w2']) / (df.loc[mask, 'epsilon'] * df.loc[mask, 'RL_qvd_w2'] + 0.5 * (df.loc[mask, 'qv2'] / df.loc[mask, 'Q2']) * df.loc[mask, 'RT_qvd_w2'])
    print('RL RT bc_qv_w2 done.')

    # calculate bc_q2_ex
    df[['Q2center','Excenter_q2']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_q2_ex.exe', 'input.txt'], stdout=output_file) 
    subprocess.run(['sleep', '0.5'])
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_q2c_ex'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_q2c_ex'] = df.index.map(df_response.set_index('i')['RTTOT'])

    df[['Q2','Ex']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_q2_ex.exe', 'input.txt'], stdout=output_file) 
    subprocess.run(['sleep', '0.5'])
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_q2d_ex'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_q2d_ex'] = df.index.map(df_response.set_index('i')['RTTOT'])

    df['bc_q2_ex'] = 1.0
    for Q2center in Q2centers:
        # Ex < 30MeV:
        mask = (df['Q2center'] == Q2center) & (df['Ex'] < Ex_cut)
        df.loc[mask, 'bc_q2_ex'] = (df.loc[mask, 'epcenter_ex_q2'] * df.loc[mask, 'RL_q2c_ex'] + 0.5 * ((Q2center + df.loc[mask, 'nucenter_ex_q2']**2) / Q2center) * df.loc[mask, 'RT_q2c_ex']) / (df.loc[mask, 'epsilon'] * df.loc[mask, 'RL_q2d_ex'] + 0.5 * (df.loc[mask, 'qv2'] / df.loc[mask, 'Q2']) * df.loc[mask, 'RT_q2d_ex'])
    print('RL RT bc_q2_ex done.')

    # calculate bc_q2_w2
    df[['Q2center','W2center_q2']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_q2_w2.exe', 'input.txt'], stdout=output_file) 
    subprocess.run(['sleep', '0.5'])
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_q2c_w2'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_q2c_w2'] = df.index.map(df_response.set_index('i')['RTTOT'])

    df[['Q2','W2']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_q2_w2.exe', 'input.txt'], stdout=output_file) 
    subprocess.run(['sleep', '0.5'])
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_q2d_w2'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_q2d_w2'] = df.index.map(df_response.set_index('i')['RTTOT'])

    df['bc_q2_w2'] = 1.0
    for Q2center in Q2centers:
        # Ex >= 30MeV:
        mask = (df['Q2center'] == Q2center) & (df['Ex'] >= Ex_cut)
        df.loc[mask, 'bc_q2_w2'] = (df.loc[mask, 'epcenter_w2_q2'] * df.loc[mask, 'RL_q2c_w2'] + 0.5 * ((Q2center + df.loc[mask, 'nucenter_w2_q2']**2) / Q2center) * df.loc[mask, 'RT_q2c_w2']) / (df.loc[mask, 'epsilon'] * df.loc[mask, 'RL_q2d_w2'] + 0.5 * (df.loc[mask, 'qv2'] / df.loc[mask, 'Q2']) * df.loc[mask, 'RT_q2d_w2'])
    print('RL RT bc_q2_w2 done.')

    return df

<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:17: SyntaxWarning: invalid escape sequence '\s'
<>:35: SyntaxWarning: invalid escape sequence '\s'
<>:43: SyntaxWarning: invalid escape sequence '\s'
<>:59: SyntaxWarning: invalid escape sequence '\s'
<>:67: SyntaxWarning: invalid escape sequence '\s'
<>:83: SyntaxWarning: invalid escape sequence '\s'
<>:91: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:17: SyntaxWarning: invalid escape sequence '\s'
<>:35: SyntaxWarning: invalid escape sequence '\s'
<>:43: SyntaxWarning: invalid escape sequence '\s'
<>:59: SyntaxWarning: invalid escape sequence '\s'
<>:67: SyntaxWarning: invalid escape sequence '\s'
<>:83: SyntaxWarning: invalid escape sequence '\s'
<>:91: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Rhys\AppData\Local\Temp\ipykernel_19796\2059628725.py:9: SyntaxWarning: invalid escape sequence '\s'
  df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=resp

In [ ]:
df = pd.read_csv('Data/C12.csv')
df = prepare_df(df)
df = calculate_bc(df)
df.to_csv('Data/df_C12.csv',index=False)
df

C:\Users\Rhys\AppData\Local\Temp\ipykernel_19796\3460560264.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Exbin_qv'] = 0.0
C:\Users\Rhys\AppData\Local\Temp\ipykernel_19796\3460560264.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Excenter_qv'] = 0.0
C:\Users\Rhys\AppData\Local\Temp\ipykernel_19796\3460560264.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the do

[-1]
RL RT bc_q2_ex done.
RL RT bc_qv_w2 done.
RL RT bc_q2_ex done.
RL RT bc_q2_w2 done.


,Z,A,E0,ThetaDeg,nu,cross,error,dataSet,normalization,normError,...,RL_q2c_ex,RT_q2c_ex,RL_q2d_ex,RT_q2d_ex,bc_q2_ex,RL_q2c_w2,RT_q2c_w2,RL_q2d_w2,RT_q2d_w2,bc_q2_w2
1,6,12,1.1111,37.5,0.016620,0.141794,0.141794,-1,1.0,0.0,...,0.000000,0.000000e+00,0.000000e+00,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,1.000000
2,6,12,1.1111,37.5,0.027700,1.985120,0.530546,-1,1.0,0.0,...,0.000000,0.000000e+00,0.000000e+00,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,1.000000
3,6,12,1.1111,37.5,0.038780,11.343500,1.268250,-1,1.0,0.0,...,0.000002,1.712931e-05,7.269416e-07,0.000006,3.066620,0.000000,0.000000,0.000002,0.000014,1.000000
4,6,12,1.1111,37.5,0.049860,27.508100,1.974970,-1,1.0,0.0,...,0.000016,1.393453e-04,1.936363e-05,0.000159,0.869476,0.000000,0.000000,0.000020,0.000163,1.000000
5,6,12,1.1111,37.5,0.060940,50.762400,2.682880,-1,1.0,0.0,...,0.000000,0.000000e+00,3.305967e-05,0.000280,1.000000,0.000041,0.000374,0.000034,0.000288,1.288965
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26482,6,12,0.9641,37.5,0.879315,3163.910000,20.952600,-1,1.0,0.0,...,0.011550,4.082654e-09,8.442584e-02,0.036093,1.000000,0.120903,0.037003,0.088021,0.036390,1.268360
26483,6,12,0.9641,37.5,0.888925,3199.710000,21.070800,-1,1.0,0.0,...,0.011550,4.082654e-09,1.025982e-01,0.036729,1.000000,0.120903,0.037003,0.107030,0.036923,1.092883
26484,6,12,0.9641,37.5,0.898535,3257.160000,21.259100,-1,1.0,0.0,...,0.011550,4.082654e-09,1.255941e-01,0.037267,1.000000,0.137767,0.037687,0.131073,0.037441,1.046104
26485,6,12,0.9641,37.5,0.908145,2459.310000,18.472800,-1,1.0,0.0,...,0.008393,1.025791e-08,1.548525e-01,0.037789,1.000000,0.164504,0.037760,0.161662,0.037944,1.022868
